# Atlas Fulfillment: Example Analysis

This notebook walks through the synthetic `atlas-fulfillment` running example that ships with `oceldb`.

The dataset models a multi-object fulfillment and reverse-logistics process with these object types:

- `customer`
- `order`
- `order_item`
- `package`
- `shipment`
- `invoice`
- `payment`
- `return_case`
- `supplier_order`

The process includes fraud review, backorders, supplier replenishment, split packing, split shipping, customs holds, delayed payment, returns, inspection, restocking, scrapping, and refunds.

The notebook keeps the main analysis path fast and uses only the public `oceldb` API. A final optional section shows heavier relation-based predicates that are useful, but noticeably slower on larger logs.

In [ ]:
from pathlib import Path
from pprint import pprint

from oceldb import OCEL, asc, avg, col, count, desc


def find_dataset() -> Path:
    relative_candidates = [
        Path("examples/data/atlas-fulfillment"),
        Path("data/atlas-fulfillment"),
        Path("atlas-fulfillment"),
    ]

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        for relative in relative_candidates:
            candidate = (root / relative).resolve()
            if candidate.exists():
                return candidate

    tried = [str((root / relative).resolve()) for root in search_roots for relative in relative_candidates]
    raise FileNotFoundError("Could not locate atlas-fulfillment. Tried:\n" + "\n".join(tried))


def _format_value(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)


def show_rows(headers: tuple[str, ...], rows: list[tuple[object, ...]], *, limit: int | None = None) -> None:
    if limit is not None:
        rows = rows[:limit]

    if not rows:
        print("(no rows)")
        return

    widths = [len(header) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(_format_value(value)))

    header_line = " | ".join(header.ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(divider)
    for row in rows:
        print(" | ".join(_format_value(value).ljust(widths[index]) for index, value in enumerate(row)))


## Load The Running Example

This cell locates the dataset directory, opens it with `OCEL.read(...)`, and keeps the handle open for the rest of the notebook.

In [ ]:
SOURCE = find_dataset()

try:
    ocel.close()
except Exception:
    pass

ocel = OCEL.read(SOURCE)
print(f"Loaded: {SOURCE}")
print(ocel)
print(ocel.manifest)

## Structural Overview

Start with the built-in inspection layer. These helpers are implemented on top of the DSL and give a quick feel for scale and object-centric structure.

In [ ]:
overview = ocel.inspect.overview()
event_object_stats = ocel.inspect.event_object_stats()

overview, event_object_stats

In [ ]:
event_type_counts = ocel.inspect.event_type_counts()
object_type_counts = ocel.inspect.object_type_counts()

print("Event type counts")
pprint(event_type_counts)
print()
print("Object type counts")
pprint(object_type_counts)

In [ ]:
attribute_map = ocel.inspect.attributes()
selected_attributes = {
    "object": {
        "order": attribute_map["object"]["order"],
        "shipment": attribute_map["object"]["shipment"],
        "return_case": attribute_map["object"]["return_case"],
    },
    "event": {
        "Place Order": attribute_map["event"]["Place Order"],
        "Deliver Shipment": attribute_map["event"]["Deliver Shipment"],
        "Issue Refund": attribute_map["event"]["Issue Refund"],
    },
}

pprint(selected_attributes)

## Event Activity

The first analytical pass is usually event-centric: which event types dominate the log and how activity evolves over time.

In [ ]:
top_event_types = (
    ocel.query
    .events()
    .group_by("ocel_type")
    .agg(count().alias("events"))
    .sort(desc("events"), asc("ocel_type"))
    .collect()
    .fetchall()
)

show_rows(("event_type", "events"), top_event_types, limit=15)

In [ ]:
delivery_quality = (
    ocel.query
    .events("Deliver Shipment")
    .group_by("on_time")
    .agg(
        count().alias("shipments"),
        avg(col("delivery_days")).alias("avg_delivery_days"),
    )
    .sort(desc("shipments"))
    .collect()
    .fetchall()
)

show_rows(("on_time", "shipments", "avg_delivery_days"), delivery_quality)

In [ ]:
monthly_event_profile = ocel.sql(
    f'''
    SELECT date_trunc('month', ocel_time) AS month,
           ocel_type,
           COUNT(*) AS events
    FROM "event"
    GROUP BY 1, 2
    ORDER BY month, events DESC, ocel_type
    LIMIT 18
    '''
).fetchall()

show_rows(("month", "event_type", "events"), monthly_event_profile)

## Orders And Commercial Behavior

Now switch to object-rooted analysis. Orders are the natural business anchor for commercial questions such as channel mix, payment behavior, and unusually large cases.

In [ ]:
order_status_mix = (
    ocel.query
    .object_states("order")
    .latest()
    .group_by("country", "channel", "status")
    .agg(
        count().alias("orders"),
        avg(col("order_value")).alias("avg_order_value"),
    )
    .sort(desc("orders"), desc("avg_order_value"))
    .limit(15)
    .collect()
    .fetchall()
)

show_rows(
    ("country", "channel", "status", "orders", "avg_order_value"),
    order_status_mix,
)

In [ ]:
payment_mix = (
    ocel.query
    .object_states("payment")
    .latest()
    .group_by("method", "status")
    .agg(
        count().alias("payments"),
        avg(col("amount")).alias("avg_amount"),
    )
    .sort(desc("payments"), desc("avg_amount"))
    .collect()
    .fetchall()
)

show_rows(("method", "status", "payments", "avg_amount"), payment_mix)

In [ ]:
high_value_orders = (
    ocel.query
    .object_states("order")
    .latest()
    .select(
        "ocel_id",
        "country",
        "channel",
        "priority",
        "risk_bucket",
        "payment_method",
        "order_value",
        "status",
    )
    .sort(desc("order_value"))
    .limit(15)
    .collect()
    .fetchall()
)

show_rows(
    (
        "order_id",
        "country",
        "channel",
        "priority",
        "risk_bucket",
        "payment_method",
        "order_value",
        "status",
    ),
    high_value_orders,
)

## Operations And Exceptions

The same dataset also supports operational analysis around replenishment, shipping complexity, and returns.

In [ ]:
supplier_pressure = (
    ocel.query
    .object_states("supplier_order")
    .latest()
    .group_by("supplier", "status")
    .agg(
        count().alias("supplier_orders"),
        avg(col("lead_time_days")).alias("avg_lead_time_days"),
    )
    .sort(desc("supplier_orders"), desc("avg_lead_time_days"))
    .collect()
    .fetchall()
)

show_rows(
    ("supplier", "status", "supplier_orders", "avg_lead_time_days"),
    supplier_pressure,
)

In [ ]:
shipment_mix = (
    ocel.query
    .object_states("shipment")
    .latest()
    .group_by("carrier", "is_international", "had_customs_hold")
    .agg(
        count().alias("shipments"),
        avg(col("attempt_count")).alias("avg_attempt_count"),
    )
    .sort(desc("shipments"), desc("avg_attempt_count"))
    .collect()
    .fetchall()
)

show_rows(
    ("carrier", "is_international", "had_customs_hold", "shipments", "avg_attempt_count"),
    shipment_mix,
)

In [ ]:
return_mix = (
    ocel.query
    .object_states("return_case")
    .latest()
    .group_by("reason", "resolution")
    .agg(
        count().alias("cases"),
        avg(col("refundable_amount")).alias("avg_refundable_amount"),
    )
    .sort(desc("cases"), desc("avg_refundable_amount"))
    .collect()
    .fetchall()
)

show_rows(
    ("reason", "resolution", "cases", "avg_refundable_amount"),
    return_mix,
)

## Relation Graph View With Raw SQL

The DSL is the preferred interface for most analysis, but `ocel.sql(...)` is useful for advanced structural summaries that are easier to express as joins. The query below shows which object-type pairs dominate the link graph.

In [ ]:
link_graph = ocel.sql(
    f'''
    SELECT source.ocel_type AS source_type,
           target.ocel_type AS target_type,
           COUNT(*) AS links
    FROM "object_object" oo
    JOIN "object" source
      ON oo.ocel_source_id = source.ocel_id
    JOIN "object" target
      ON oo.ocel_target_id = target.ocel_id
    GROUP BY 1, 2
    ORDER BY links DESC, source_type, target_type
    LIMIT 15
    '''
).fetchall()

show_rows(("source_type", "target_type", "links"), link_graph)

## Optional: Object-Centric Slices With Relation Builders

This final section demonstrates the more OCEL-specific part of the DSL with free relation builders such as `linked(...)` and `has_event(...)`.

These predicates are correlated and can be noticeably slower than the aggregate sections above, especially if you run several of them in one go. Keep them for targeted questions rather than the default analysis path.

In [ ]:
from time import perf_counter

from oceldb import has_event, linked


def timed(label: str, fn):
    start = perf_counter()
    value = fn()
    elapsed = perf_counter() - start
    print(f"{label}: {elapsed:.2f}s -> {value}")
    return value


orders = ocel.query.objects("order")


slow_relation_summary = {
    "orders_linked_to_supplier_orders": timed(
        "orders linked to supplier orders",
        lambda: orders.where(linked("supplier_order").exists()).count(),
    ),
    "orders_linked_to_returns": timed(
        "orders linked to return cases",
        lambda: orders.where(linked("return_case").exists()).count(),
    ),
    "orders_with_customs_hold_event": timed(
        "orders with customs hold",
        lambda: orders.where(has_event("Customs Hold").exists()).count(),
    ),
}

slow_relation_summary

## Close The Handle

Close the `OCEL` once you are done. Re-running the load cell will open a fresh handle.

In [ ]:
ocel.close()